# Chronos-2 Forecasting — DIMER live tutorial

This notebook runs the repository's pinned Amazon Chronos-2 pipeline end-to-end: deterministic sample or BYOD input, chronological holdout, zero-shot forecast, uncertainty interval, leakage-safe evaluation, naive baselines, and portable forecast/provenance exports.

**No fine-tuning occurs.** The point forecast is the model's median (`q0.5`), not a statistical mean. The default path uses CPU and a 12-step horizon so it is suitable for a fresh Colab session.


## 1. Bootstrap the repository and locked runtime

When opened from GitHub in Colab, this cell clones the public repository, installs the exact exported dependency graph with `uv`, and installs the local package without resolving a second dependency graph. Under repository CI, the checkout and locked environment already exist, so the cell is a no-op.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline.git"
REPO_NAME = "chronos-2-forecasting-pipeline"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(
        ["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--no-deps", "-e", "."],
        check=True,
    )
else:
    print(f"Repository checkout detected: {ROOT}")


## 2. Choose the bundled sample or BYOD

The checked-in default sample is synthetic and deterministic; its provenance and SHA-256 are documented in `examples/sample-data/DATASET_CARD.md`.

Set `USE_BYOD = True` in Colab to upload a CSV with `series_id`, `timestamp`, and `target` columns. The pipeline rejects duplicate/gappy/irregular series rather than silently interpolating them.


In [ ]:
import io
import json

import pandas as pd

from chronos2_pipeline import (
    ForecastConfig,
    chronological_holdout,
    evaluate_forecast,
    forecast,
    last_value_baseline,
    load_pinned_model,
    seasonal_naive_baseline,
)

USE_BYOD = False
PREDICTION_LENGTH = 12

if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("BYOD upload is available when this notebook runs in Colab.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV for the beginner BYOD path.")
    name, payload = next(iter(uploaded.items()))
    frame = pd.read_csv(io.BytesIO(payload))
    print(f"Loaded BYOD: {name}")
else:
    sample_path = ROOT / "examples" / "sample-data" / "chronos_univariate.csv"
    frame = pd.read_csv(sample_path)
    print(f"Loaded bundled sample: {sample_path}")

frame["timestamp"] = pd.to_datetime(frame["timestamp"])
print(frame.head())
print(f"rows={len(frame)}, ids={frame['series_id'].nunique()}")


## 3. Hold out the future chronologically

For demonstration only, the final 12 timestamps are removed before inference and retained as truth. The model never sees those target values. For genuine future forecasting, skip the holdout and forecast directly from all available history.


In [ ]:
config = ForecastConfig(
    target="target",
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.1, 0.5, 0.9],
    device="cpu",
)

split = chronological_holdout(frame, config)
print(
    "context through:",
    split.history["timestamp"].max(),
    "| held-out future starts:",
    split.truth["timestamp"].min(),
)
assert split.history["timestamp"].max() < split.truth["timestamp"].min()


### History visualization

The tutorial writes a dependency-free SVG so the locked model runtime does not need a plotting package. Colab/IPython displays it inline when available.


In [ ]:
def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    if not all_values:
        raise ValueError("cannot plot an empty series")
    low = min(all_values)
    high = max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38

    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"

    strokes = ["#111827", "#2563eb", "#dc2626", "#059669"]
    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>',
        f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#9ca3af"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#9ca3af"/>',
    ]
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = strokes[idx % len(strokes)]
        svg.append(
            f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>'
        )
        svg.append(
            f'<text x="{left + 120 * idx}" y="{height - 10}" '
            f'font-family="sans-serif" font-size="12" fill="{stroke}">{label}</text>'
        )
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

history_svg = write_line_svg(
    ROOT / "outputs" / "chronos_history.svg",
    [("history", split.history["target"].tail(48).tolist())],
    title="Chronos-2 tutorial: recent history",
)
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(history_svg)))
except ImportError:
    print(f"History SVG written to {history_svg}")


## 4. Resolve and verify the pinned Chronos-2 checkpoint

The loader accepts only the repository's approved `amazon/chronos-2` revision, checks the pinned model/config digests, and refuses pickle-format fallback. The first run downloads roughly 478 MB.


In [ ]:
model = load_pinned_model(device="cpu")
print("model:", model.identity.model_id)
print("revision:", model.identity.revision)
print("weights sha256:", model.identity.weights_sha256)
print("device/dtype:", model.device, model.dtype)


## 5. Run the zero-shot forecast


In [ ]:
result = forecast(split.history, config, model)
print(result.forecast.head())
print("latency_seconds:", result.inference["latency_seconds"])
print("effective_context_length:", result.inference["effective_context_length"])


## 6. Evaluate the held-out future and compare naive baselines

Metrics are computed on the identical held-out timestamps. The evaluator reports point MAE/RMSE, quantile pinball loss, and empirical coverage of the outer requested interval. The baseline uses history only.


In [ ]:
evaluation = evaluate_forecast(result.forecast, split.truth, config)

last_value = last_value_baseline(split.history, split.truth, config)
last_value_evaluation = evaluate_forecast(last_value, split.truth, config)

seasonal = seasonal_naive_baseline(
    split.history,
    split.truth,
    config,
    season_length=24,
)
seasonal_evaluation = evaluate_forecast(seasonal, split.truth, config)

print("Chronos-2:", evaluation.aggregate)
print("Last-value:", last_value_evaluation.aggregate)
print("Seasonal-naive (24):", seasonal_evaluation.aggregate)
print(evaluation.quantiles)


## 7. Visualize the median and 80% model interval

`prediction` is the median (`q0.5`). `q0.1` and `q0.9` are model quantiles, not guaranteed calibration bounds for a new domain.


In [ ]:
forecast_frame = result.forecast.sort_values("timestamp")
truth_frame = split.truth.sort_values("timestamp")

forecast_svg = write_line_svg(
    ROOT / "outputs" / "chronos_forecast.svg",
    [
        ("q0.1", forecast_frame["q0.1"].tolist()),
        ("median", forecast_frame["prediction"].tolist()),
        ("q0.9", forecast_frame["q0.9"].tolist()),
        ("truth", truth_frame["target"].tolist()),
    ],
    title="Held-out future: Chronos-2 quantiles and truth",
)
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(forecast_svg)))
except ImportError:
    print(f"Forecast SVG written to {forecast_svg}")


## 8. Export forecast, evaluation and provenance

The CSV contains the normalized DIMER forecast contract. The JSON preserves the immutable model identity, runtime versions, requested/effective context and horizon, quantiles, and latency metadata.


In [ ]:
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

forecast_path = OUTPUT_DIR / "chronos_forecast.csv"
provenance_path = OUTPUT_DIR / "chronos_provenance.json"
metrics_path = OUTPUT_DIR / "chronos_evaluation.json"
per_series_path = OUTPUT_DIR / "chronos_evaluation_per_series.csv"

result.forecast.to_csv(forecast_path, index=False)
per_series_path.write_text(evaluation.per_series.to_csv(index=False), encoding="utf-8")
provenance_path.write_text(
    json.dumps(result.provenance, indent=2, default=str),
    encoding="utf-8",
)
metrics_path.write_text(
    json.dumps(
        {
            "chronos2": evaluation.aggregate,
            "last_value": last_value_evaluation.aggregate,
            "seasonal_naive_24": seasonal_evaluation.aggregate,
            "quantiles": evaluation.quantiles.to_dict("records"),
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("wrote:", forecast_path)
print("wrote:", provenance_path)
print("wrote:", metrics_path)
print("wrote:", per_series_path)


## Optional: Mode D known-future covariates

The sample generator also creates `chronos_covariates_history.csv` and `chronos_covariates_future.csv`. The future table contains only `temperature` and `holiday`—never future `demand`. Set the switch below to run the optional demonstration.


In [ ]:
RUN_COVARIATE_DEMO = False

if RUN_COVARIATE_DEMO:
    generator = ROOT / "examples" / "sample-data" / "generate_samples.py"
    subprocess.run([sys.executable, str(generator)], check=True)
    cov_history = pd.read_csv(
        ROOT / "examples" / "sample-data" / "chronos_covariates_history.csv"
    )
    cov_future = pd.read_csv(
        ROOT / "examples" / "sample-data" / "chronos_covariates_future.csv"
    )
    cov_config = ForecastConfig(
        target="demand",
        prediction_length=24,
        quantile_levels=[0.1, 0.5, 0.9],
        device="cpu",
    )
    cov_result = forecast(cov_history, cov_config, model, cov_future)
    print(cov_result.forecast.head())
    print(cov_result.inference["known_future_covariate_names"])


## Interpretation and limits

- This is zero-shot inference; nothing in the notebook fine-tunes Chronos-2.
- Tutorial metrics are pedagogical, not an unbiased benchmark claim. The synthetic sample is intentionally simple.
- Quantiles are not guaranteed calibrated on your data.
- Fixed-width regular frequencies are supported; monthly/quarterly/yearly/business-day calendars remain outside the current contract.
- BYOD data should be evaluated with domain-appropriate baselines and leakage controls before operational use.
